# NLP Applied to Emotion Detection — Mini-Project
## SLM vs LLM for Emotion Detection

**Author:** ValYu777

---

This notebook investigates the trade-offs between a fine-tuned Small Language Model (DistilBERT) and a Large Language Model (Mistral-7B) for 6-class emotion detection on the `dair-ai/emotion` dataset.

## Section 1 — Setup

In [ ]:
import time
import re
import json
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict

from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("All imports OK.")

All imports OK.


In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
dataset = load_dataset("dair-ai/emotion")
test_ds  = dataset["test"]
train_ds = dataset["train"]

# Label mapping (same order as the HuggingFace model)
LABEL_NAMES = ["sadness", "joy", "love", "anger", "fear", "surprise"]
LABEL_IDS   = {name: idx for idx, name in enumerate(LABEL_NAMES)}

print(f"Test set size : {len(test_ds)}")
print(f"Label mapping : {LABEL_IDS}")
test_ds[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Test set size : 2000
Label mapping : {'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}


{'text': 'im feeling rather rotten so im not very ambitious right now',
 'label': 0}

In [ ]:
# ── NOTE: LLM API ────────────────────────────────────────────────────────────
# We originally intended to use the HuggingFace Inference API with Mistral-7B,
# but switched to the Groq API (llama-3.3-70b-versatile) because:
#   • The HuggingFace free tier frequently returned 503 errors (model loading).
#   • Groq provides a faster, more reliable free inference endpoint.
# llama-3.3-70b-versatile is a Llama-based instruction-tuned LLM comparable
# in capability to Mistral-7B-Instruct for this classification task.


In [ ]:
!pip install groq -q
from groq import Groq

GROQ_API_KEY = "YOUR_API_KEY_HERE"

# Model used as the LLM throughout this notebook.
# llama-3.3-70b-versatile is a large instruction-tuned model served for free
# via Groq's low-latency inference infrastructure.
HF_MODEL = "llama-3.3-70b-versatile"

groq_client = Groq(api_key=GROQ_API_KEY)
print(f"LLM backend : Groq")
print(f"Model       : {HF_MODEL}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.0 MB/s eta 0:00:00
LLM backend : Groq
Model       : llama-3.3-70b-versatile


In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────
# call_mistral() wraps the Groq API (llama-3.3-70b-versatile).
# The function name is kept as 'call_mistral' for consistency with the
# rest of the notebook, but the underlying call goes to Groq.
#
# parse_emotion()            → scans raw text for any of the 6 label names.
# parse_emotion_after_answer() → for CoT prompts, looks for 'Answer: <label>'.
# stratified_sample()        → returns n_per_class examples for each label.

# ── Helper functions ──────────────────────────────────────────────────────────

def call_mistral(prompt: str, temperature: float = 0.0,
                 max_new_tokens: int = 50, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            response = groq_client.chat.completions.create(
                model=HF_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=max(temperature, 0.01),
                max_tokens=max_new_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f"  Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    return ""

def parse_emotion(text: str):
    """
    Return the first emotion label found in `text` (lowercased), or None.
    """
    text_lower = text.lower()
    for label in LABEL_NAMES:
        if label in text_lower:
            return label
    return None


def parse_emotion_after_answer(text: str):
    """
    For chain-of-thought prompts: look for 'Answer: <emotion>' first,
    then fall back to parse_emotion.
    """
    import re
    match = re.search(r"answer[:\s]+([a-z]+)", text.lower())
    if match:
        word = match.group(1)
        if word in LABEL_NAMES:
            return word
    return parse_emotion(text)


def stratified_sample(dataset, n_per_class: int = 50, seed: int = 42):
    """
    Return a list of (text, label_int) tuples with exactly
    `n_per_class` examples per class.
    """
    rng = np.random.default_rng(seed)
    buckets = defaultdict(list)
    for ex in dataset:
        buckets[ex["label"]].append(ex)
    samples = []
    for label, items in sorted(buckets.items()):
        idx = rng.choice(len(items), size=n_per_class, replace=False)
        samples.extend([(items[i]["text"], items[i]["label"]) for i in idx])
    rng.shuffle(samples)
    return samples


# ── Smoke test ────────────────────────────────────────────────────────────────
resp = call_mistral("Reply with a single word. What is the capital of France?",
                    temperature=0.0, max_new_tokens=10)
print(f"Smoke test response: '{resp}'")


Smoke test response: 'Paris.'


---
## Section 2 — P.1 Baseline Evaluation

We evaluate both models on the same 6-class emotion detection task
(`sadness`, `joy`, `love`, `anger`, `fear`, `surprise`).

| Model | Backend | Evaluation set |
|---|---|---|
| **SLM** — `bhadresh-savani/distilbert-base-uncased-emotion` | Local HuggingFace pipeline | Full test set (2 000 sentences) |
| **LLM** — `llama-3.3-70b-versatile` | Groq Inference API | Stratified sample of 300 sentences (50 per class) |

The LLM is queried at `temperature=0.0` with a minimal single-word-answer prompt.
Unparseable responses (where no label name is found in the output) are excluded
before computing metrics and their count is reported separately.


In [ ]:
# ── P.1a — SLM: DistilBERT ────────────────────────────────────────────────────
slm_pipe = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    top_k=1,
)

test_texts  = list(test_ds["text"])   # cast Arrow array → plain Python list
test_labels = list(test_ds["label"])

# Batched inference — fast on CPU too
raw_preds = slm_pipe(test_texts, batch_size=64, truncation=True)

# raw_preds is a list of lists: [[{'label': 'joy', 'score': 0.98}], ...]
slm_pred_labels = [r[0]["label"].lower() for r in raw_preds]
slm_pred_ids    = [LABEL_IDS[l] for l in slm_pred_labels]

slm_acc = accuracy_score(test_labels, slm_pred_ids)
slm_f1  = f1_score(test_labels, slm_pred_ids, average="macro")

print(f"SLM  Accuracy : {slm_acc:.4f}")
print(f"SLM  Macro F1 : {slm_f1:.4f}")
print()
print(classification_report(test_labels, slm_pred_ids, target_names=LABEL_NAMES))

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

SLM  Accuracy : 0.9270
SLM  Macro F1 : 0.8825

              precision    recall  f1-score   support

     sadness       0.97      0.97      0.97       581
         joy       0.95      0.94      0.95       695
        love       0.80      0.85      0.83       159
       anger       0.93      0.92      0.92       275
        fear       0.88      0.92      0.90       224
    surprise       0.80      0.68      0.74        66

    accuracy                           0.93      2000
   macro avg       0.89      0.88      0.88      2000
weighted avg       0.93      0.93      0.93      2000



In [ ]:
# ── P.1b — LLM: llama-3.3-70b-versatile on 300-sentence stratified sample ──

PROMPT_A = """Classify the emotion expressed in the following sentence.
Choose exactly one from: sadness, joy, love, anger, fear, surprise.
Reply with a single word — the emotion label only.

Sentence: {sentence}
"""

sample_300 = stratified_sample(test_ds, n_per_class=50, seed=42)

llm_p1_texts, llm_p1_true = zip(*sample_300)

llm_p1_raw    = []
llm_p1_parsed = []

for text in tqdm(llm_p1_texts, desc="LLM P.1"):
    prompt   = PROMPT_A.format(sentence=text)
    raw      = call_mistral(prompt, temperature=0.0)
    parsed   = parse_emotion(raw)
    llm_p1_raw.append(raw)
    llm_p1_parsed.append(parsed)
    time.sleep(1)

n_unparseable = llm_p1_parsed.count(None)
print(f"Unparseable responses: {n_unparseable} / {len(llm_p1_parsed)}")

# Exclude unparseable before computing metrics
mask = [p is not None for p in llm_p1_parsed]
llm_p1_pred_ids = [LABEL_IDS[p] for p, m in zip(llm_p1_parsed, mask) if m]
llm_p1_true_flt = [t          for t, m in zip(llm_p1_true,   mask) if m]

llm_acc = accuracy_score(llm_p1_true_flt, llm_p1_pred_ids)
llm_f1  = f1_score(llm_p1_true_flt, llm_p1_pred_ids, average="macro")

print(f"LLM  Accuracy : {llm_acc:.4f}")
print(f"LLM  Macro F1 : {llm_f1:.4f}")
print()
print(classification_report(llm_p1_true_flt, llm_p1_pred_ids, target_names=LABEL_NAMES))

LLM P.1:   0%|          | 0/300 [00:00<?, ?it/s]

Unparseable responses: 0 / 300
LLM  Accuracy : 0.4400
LLM  Macro F1 : 0.4350

              precision    recall  f1-score   support

     sadness       0.34      0.66      0.45        50
         joy       0.35      0.48      0.41        50
        love       0.45      0.18      0.26        50
       anger       0.59      0.44      0.51        50
        fear       0.59      0.44      0.51        50
    surprise       0.55      0.44      0.49        50

    accuracy                           0.44       300
   macro avg       0.48      0.44      0.44       300
weighted avg       0.48      0.44      0.44       300



In [ ]:
# ── P.1c — Summary DataFrame ──────────────────────────────────────────────────
from sklearn.metrics import precision_recall_fscore_support

# Per-class F1 for SLM
_, _, slm_f1_per_class, _ = precision_recall_fscore_support(
    test_labels, slm_pred_ids, labels=list(range(6)), zero_division=0)

# Per-class F1 for LLM
_, _, llm_f1_per_class, _ = precision_recall_fscore_support(
    llm_p1_true_flt, llm_p1_pred_ids, labels=list(range(6)), zero_division=0)

comparison_df = pd.DataFrame({
    "Emotion"     : LABEL_NAMES,
    "SLM F1"      : slm_f1_per_class.round(3),
    "LLM F1"      : llm_f1_per_class.round(3),
    "Δ (LLM−SLM)" : (llm_f1_per_class - slm_f1_per_class).round(3),
})

summary_row = pd.DataFrame([{
    "Emotion"     : "OVERALL (macro)",
    "SLM F1"      : round(slm_f1, 3),
    "LLM F1"      : round(llm_f1, 3),
    "Δ (LLM−SLM)" : round(llm_f1 - slm_f1, 3),
}])

comparison_df = pd.concat([comparison_df, summary_row], ignore_index=True)
print(comparison_df.to_string(index=False))

        Emotion  SLM F1  LLM F1  Δ (LLM−SLM)
        sadness   0.967   0.446       -0.521
            joy   0.946   0.407       -0.539
           love   0.826   0.257       -0.569
          anger   0.923   0.506       -0.418
           fear   0.895   0.506       -0.389
       surprise   0.738   0.489       -0.249
OVERALL (macro)   0.883   0.435       -0.447


### P.1 — Analysis

**Which model performed better overall?**
The SLM achieved an accuracy of **0.9270** and a macro F1 of **0.8825** on the full test
set of 2 000 sentences. The LLM reached an accuracy of **0.4400** and a macro F1 of
**0.4350** on the 300-sentence stratified sample a gap of more than 49 percentage points
in accuracy and 45 points in macro F1. This result is expected: DistilBERT was explicitly
fine-tuned on the `dair-ai/emotion` corpus and has learned a task-specific decision
boundary, whereas the LLM must infer the task entirely from a short prompt, with no
task-specific training signal.

**Were there specific classes where one model was clearly stronger?**
The SLM dominates across all six classes, but the gap is not uniform. The largest
differences in F1 appear on `love` (SLM: 0.826 vs LLM: 0.257, Δ = −0.569) and `joy`
(SLM: 0.946 vs LLM: 0.407, Δ = −0.539), suggesting the LLM struggles to distinguish
closely related positive emotions. The smallest gap is on `surprise` (Δ = −0.249), a class
where even the SLM scores only 0.738, reflecting the limited number of test examples (66).
The LLM performs relatively best on `fear` and `anger` (F1: 0.506 each) emotions whose
lexical signals tend to be strong and salient in general text, making them easier to
capture through zero-shot prompting.

**What might explain the difference?**
DistilBERT's representations are calibrated to this exact six-class taxonomy through
supervised fine-tuning, giving it a precise decision boundary for each label. The LLM,
guided only by a minimal one-line prompt, relies on general pre-training knowledge and
tends to over-predict broader, more frequent categories at the expense of nuanced ones:
its recall on `love` is only 0.18, while its recall on `sadness` reaches 0.66, indicating
systematic confusion between the two. These weaknesses will be probed further in the
error analysis (P.2) and prompt sensitivity experiments (P.4).

---
## Section 3 — P.2 Error Analysis

We manually inspect up to 10 misclassified examples per model and classify each error into one of:
`Negation`, `Irony/sarcasm`, `Ambiguity`, `Intensity`, `Informal/emoji`, `Other`.

In [ ]:
# ── P.2a — Collect SLM errors ─────────────────────────────────────────────────
slm_errors = [
    {"text": text, "true": LABEL_NAMES[true], "pred": pred_label}
    for text, true, pred_label in zip(test_texts, test_labels, slm_pred_labels)
    if LABEL_IDS[pred_label] != true
]
print(f"Total SLM errors: {len(slm_errors)}")

# Sample up to 10 for manual inspection — choose variety of error types
slm_sample_errors = slm_errors[:10]   # Replace with a curated selection after running
pd.DataFrame(slm_sample_errors)

Total SLM errors: 146


,text,true,pred
0,i don t feel particularly agitated,fear,anger
1,i feel if i completely hated things i d exerci...,anger,sadness
2,i feel a bit stressed even though all the thin...,anger,sadness
3,i am right handed however i play billiards lef...,surprise,fear
4,i feel like i am in paradise kissing those swe...,joy,love
5,when a friend dropped a frog down my neck,anger,fear
6,i looked at mabel this morning i named my left...,fear,surprise
7,i feel very mislead by someone that i really r...,love,anger
8,im feeling generous today heres one more you m...,love,joy
9,i actually feel agitated which led to a terrib...,anger,fear


In [ ]:
# ── P.2b — Collect LLM errors ─────────────────────────────────────────────────
llm_errors = [
    {"text": text, "true": LABEL_NAMES[true], "pred": pred}
    for text, true, pred in zip(llm_p1_texts, llm_p1_true, llm_p1_parsed)
    if pred is not None and LABEL_IDS[pred] != true
]
print(f"Total LLM errors: {len(llm_errors)}")

llm_sample_errors = llm_errors[:10]
pd.DataFrame(llm_sample_errors)

Total LLM errors: 168


,text,true,pred
0,i feel like if people accepted that wed get al...,love,sadness
1,i feel a little mellow today,joy,sadness
2,i feel a bit like a naughty kid who went and s...,love,sadness
3,i then realized that if i want to shoot weddin...,joy,love
4,im not sure if im more at peace with our situa...,anger,sadness
5,i found myself feeling a bit overwhelmed,surprise,fear
6,i feel very saddened that the king whom i once...,joy,sadness
7,i feel so blessed and honored that we get to b...,love,joy
8,i was feeling pretty anxious all day but my fi...,fear,joy
9,im feeling and if ive liked being pregnant,love,joy


In [ ]:
# ── P.2c — Manual error categorisation ───────────────────────────────────────
# Fill in 'category' after inspecting each example.
# Valid values: Negation | Irony/sarcasm | Ambiguity | Intensity | Informal/emoji | Other

# first step we take previous data, and we write category other everywhere
slm_categorised = [
    {
        "text": err["text"],
        "true": err["true"],
        "pred": err["pred"],
        "category" : "Other"
    }
    for err in slm_sample_errors
]
# second step we modify category
slm_categorised[0]["category"] = "Negation"
slm_categorised[1]["category"] = "Ambiguity"
slm_categorised[2]["category"] = "Intensity"
slm_categorised[4]["category"] = "Intensity"
slm_categorised[7]["category"] = "Irony/sarcasm"
slm_categorised[8]["category"] = "Intensity"
slm_categorised[9]["category"] = "Intensity"

llm_categorised = [
    {
        "text": err["text"],
        "true": err["true"],
        "pred": err["pred"],
        "category": "Other"
    }
    for err in llm_sample_errors
]
llm_categorised[0]["category"] = "Ambiguity"
llm_categorised[1]["category"] = "Intensity"
llm_categorised[2]["category"] = "Ambiguity"
llm_categorised[4]["category"] = "Negation"
llm_categorised[5]["category"] = "Ambiguity"
llm_categorised[6]["category"] = "Irony/sarcasm"
llm_categorised[7]["category"] = "Intensity"
llm_categorised[9]["category"] = "Intensity"

ERROR_CATS = ["Negation", "Irony/sarcasm", "Ambiguity", "Intensity", "Informal/emoji", "Other"]

def count_categories(errors):
    counts = {cat: 0 for cat in ERROR_CATS}
    for e in errors:
        counts[e["category"]] = counts.get(e["category"], 0) + 1
    return counts

error_table = pd.DataFrame({
    "Category"  : ERROR_CATS,
    "SLM count" : [count_categories(slm_categorised).get(c, 0) for c in ERROR_CATS],
    "LLM count" : [count_categories(llm_categorised).get(c, 0) for c in ERROR_CATS],
})
print(error_table.to_string(index=False))
print(llm_categorised[0])

      Category  SLM count  LLM count
      Negation          1          1
 Irony/sarcasm          1          1
     Ambiguity          1          3
     Intensity          4          3
Informal/emoji          0          0
         Other          3          2
{'text': 'i feel like if people accepted that wed get along a lot better', 'true': 'love', 'pred': 'sadness', 'category': 'Ambiguity'}


### P.2 — Analysis

**Failure patterns of the SLM.**
DistilBERT's 10 sampled errors are dominated by **Intensity** (4/10) and **Other** (3/10),
with isolated cases of Negation, Irony/sarcasm, and Ambiguity (1 each). The most
revealing pattern is low-intensity emotion: sentence 2 *"i feel a bit stressed even
though all the things i have going on are fun"* (true: `anger`) was predicted as
`sadness`, likely because the hedged phrasing ("a bit stressed") suppresses the anger
signal. Similarly, sentence 9 *"i actually feel agitated which led to a terrible…"*
(true: `anger`) was predicted as `fear`, suggesting the model latches onto downstream
consequence words rather than the stated emotion. The single Negation error is telling:
*"i don t feel particularly agitated"* (true: `fear`) was labelled `anger`, showing that
DistilBERT can fail to integrate the negative particle when the target word ("agitated")
dominates the embedding. These results confirm that the SLM is brittle when the
emotional signal is distributed across context rather than concentrated in a single
salient lexical marker.

**Failure patterns of the LLM.**
The LLM's 10 sampled errors concentrate on **Ambiguity** (3/10) and **Intensity** (3/10),
with 2 Other, and 1 each for Negation and Irony/sarcasm. The Ambiguity errors are the
most informative: *"i feel like if people accepted that wed get along a lot better"*
(true: `love`, pred: `sadness`), *"i feel a bit like a naughty kid…"* (true: `love`,
pred: `sadness`), and *"i then realized that if i want to shoot weddings…"* (true: `joy`,
pred: `love`) all involve sentiment that is positive in intent but expressed through
indirect or hedged language forms that are harder to resolve from a zero-shot prompt
alone. The Irony/sarcasm case *"i feel very saddened that the king whom i once…"*
(true: `joy`, pred: `sadness`) shows that the LLM can be misled by surface sentiment
words even when the broader context inverts their meaning.

**What does this reveal about the underlying representations?**
Both models struggle with intensity and ambiguity, but for different reasons. The SLM
operates on a task-specific embedding space optimised for discriminating these six classes
on in-distribution data; it is fast and precise, but brittle when the emotional signal is
diffuse or requires negation handling. The LLM encodes broader pragmatic and contextual
knowledge, which helps with some nuanced cases, yet its zero-shot setup makes it
vulnerable to lexical traps in ambiguous sentences where no prompt, however clear,
fully substitutes for labelled training examples. The total error counts (SLM: 146/2 000;
LLM: 168/300) reflect the fundamental performance gap already measured in P.1.

---
## Section 4 — P.3 Reproducibility and Temperature

We run each of 10 sentences three times at `temperature=0` and three times at `temperature=0.7`
through the LLM, and three times through the SLM, to measure output stability.

In [ ]:
# ── P.3a — Select 10 representative sentences ─────────────────────────────────
# Pick two sentences per class from the test set for variety.
repro_sentences = []
rng = np.random.default_rng(99)
buckets = defaultdict(list)
for ex in test_ds:
    buckets[ex["label"]].append(ex)
for label in range(6):
    chosen = rng.choice(len(buckets[label]), size=2, replace=False)
    for i in chosen:
        repro_sentences.append((buckets[label][i]["text"], label))
repro_sentences = repro_sentences[:10]   # cap at 10

print(f"Selected {len(repro_sentences)} sentences for reproducibility test.")
for i, (t, l) in enumerate(repro_sentences):
    print(f"  [{i}] ({LABEL_NAMES[l]}) {t[:80]}")

Selected 10 sentences for reproducibility test.
  [0] (sadness) i realized that i would be sad to leave this plane so soon and that just because
  [1] (sadness) i am really hurt and i feel unimportant and that sucks
  [2] (joy) ive left the orange scented mixture white but feel free to color it if you wish
  [3] (joy) i am and always have been a very sincere nice feeling sociable compassionate hel
  [4] (love) i just want that feeling of not caring about unnecessary stuff like i felt befor
  [5] (love) i remember wanting to fit in so bad and feeling like no one liked me
  [6] (anger) i really feel for the women who have to work with these obnoxious cretins
  [7] (anger) i feel so damn agitated
  [8] (fear) i do feel apprehensive and nervous at times about how i am performing with my mo
  [9] (fear) i have moments where i just feel so overwhelmed that my eyes well up with tears


In [ ]:
# ── P.3b — SLM reproducibility (should be 100%) ───────────────────────────────
slm_repro = []
for text, true_label in repro_sentences:
    runs = [slm_pipe(text, truncation=True)[0][0]["label"].lower() for _ in range(3)]
    slm_repro.append(runs)

slm_repro_df = pd.DataFrame(
    slm_repro,
    columns=["Run 1", "Run 2", "Run 3"]
)
slm_repro_df.insert(0, "True label", [LABEL_NAMES[l] for _, l in repro_sentences])
slm_repro_df["All same?"] = slm_repro_df[["Run 1","Run 2","Run 3"]].apply(
    lambda row: row.nunique() == 1, axis=1)
slm_repro_rate = slm_repro_df["All same?"].mean()
print(f"SLM reproducibility rate: {slm_repro_rate*100:.1f}%")
slm_repro_df

SLM reproducibility rate: 100.0%


,True label,Run 1,Run 2,Run 3,All same?
0,sadness,sadness,sadness,sadness,True
1,sadness,sadness,sadness,sadness,True
2,joy,joy,joy,joy,True
3,joy,joy,joy,joy,True
4,love,love,love,love,True
5,love,love,love,love,True
6,anger,anger,anger,anger,True
7,anger,anger,anger,anger,True
8,fear,fear,fear,fear,True
9,fear,fear,fear,fear,True


In [ ]:
# ── P.3c — LLM reproducibility at temperature=0 ───────────────────────────────
llm_temp0_results = []
for text, _ in tqdm(repro_sentences, desc="LLM temp=0"):
    runs = []
    for _ in range(3):
        raw    = call_mistral(PROMPT_A.format(sentence=text), temperature=0.0)
        parsed = parse_emotion(raw)
        runs.append(parsed if parsed else "UNPARSEABLE")
        time.sleep(1)
    llm_temp0_results.append(runs)

# ── P.3d — LLM reproducibility at temperature=0.7 ────────────────────────────
llm_temp07_results = []
for text, _ in tqdm(repro_sentences, desc="LLM temp=0.7"):
    runs = []
    for _ in range(3):
        raw    = call_mistral(PROMPT_A.format(sentence=text), temperature=0.7)
        parsed = parse_emotion(raw)
        runs.append(parsed if parsed else "UNPARSEABLE")
        time.sleep(1)
    llm_temp07_results.append(runs)

LLM temp=0:   0%|          | 0/10 [00:00<?, ?it/s]

LLM temp=0.7:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
# ── P.3e — Build summary table ────────────────────────────────────────────────
repro_rows = []
for i, (text, true_label) in enumerate(repro_sentences):
    t0  = llm_temp0_results[i]
    t07 = llm_temp07_results[i]
    repro_rows.append({
        "Sentence (truncated)" : text[:60] + "...",
        "True"                 : LABEL_NAMES[true_label],
        "T=0  Run1"            : t0[0], "T=0  Run2": t0[1], "T=0  Run3": t0[2],
        "T=0  Changed?"        : len(set(t0)) > 1,
        "T=0.7 Run1"           : t07[0], "T=0.7 Run2": t07[1], "T=0.7 Run3": t07[2],
        "T=0.7 Changed?"       : len(set(t07)) > 1,
    })

repro_df = pd.DataFrame(repro_rows)

pct_change_t0  = repro_df["T=0  Changed?"].mean() * 100
pct_change_t07 = repro_df["T=0.7 Changed?"].mean() * 100

print(f"% label changes at temperature=0  : {pct_change_t0:.1f}%")
print(f"% label changes at temperature=0.7: {pct_change_t07:.1f}%")
print(f"SLM reproducibility rate           : {slm_repro_rate*100:.1f}%")
repro_df

% label changes at temperature=0  : 0.0%
% label changes at temperature=0.7: 0.0%
SLM reproducibility rate           : 100.0%


,Sentence (truncated),True,T=0 Run1,T=0 Run2,T=0 Run3,T=0 Changed?,T=0.7 Run1,T=0.7 Run2,T=0.7 Run3,T=0.7 Changed?
0,i realized that i would be sad to leave this p...,sadness,sadness,sadness,sadness,False,sadness,sadness,sadness,False
1,i am really hurt and i feel unimportant and th...,sadness,sadness,sadness,sadness,False,sadness,sadness,sadness,False
2,ive left the orange scented mixture white but ...,joy,joy,joy,joy,False,joy,joy,joy,False
3,i am and always have been a very sincere nice ...,joy,joy,joy,joy,False,joy,joy,joy,False
4,i just want that feeling of not caring about u...,love,sadness,sadness,sadness,False,sadness,sadness,sadness,False
5,i remember wanting to fit in so bad and feelin...,love,sadness,sadness,sadness,False,sadness,sadness,sadness,False
6,i really feel for the women who have to work w...,anger,anger,anger,anger,False,anger,anger,anger,False
7,i feel so damn agitated...,anger,anger,anger,anger,False,anger,anger,anger,False
8,i do feel apprehensive and nervous at times ab...,fear,fear,fear,fear,False,fear,fear,fear,False
9,i have moments where i just feel so overwhelme...,fear,sadness,sadness,sadness,False,sadness,sadness,sadness,False


### P.3 — Analysis

**What is temperature?**
Temperature controls the randomness of token sampling in an autoregressive language model.
At `temperature=0`, the model greedily selects the highest-probability token at every step,
making output near-deterministic. At `temperature=0.7`, the probability distribution over
tokens is broadened, introducing more diversity in the generated responses at the cost
of potentially higher variance.

**Does `temperature=0` guarantee reproducibility?**
In our experiment, the LLM produced identical labels across all 3 runs at both
`temperature=0` and `temperature=0.7`, yielding **0.0% label changes** in both cases. The
stability at `temperature=0` is consistent with greedy decoding, though it does not
constitute an absolute guarantee: floating-point non-determinism on GPU hardware or
load-balancing across remote server instances can theoretically introduce occasional
variations not captured by our 10-sentence sample. The stability at `temperature=0.7`
is more surprising, and reflects high model confidence: when a token's probability mass
is very concentrated, increasing entropy rarely changes the argmax outcome. On a larger
sample or with more ambiguous sentences, variation at `temperature=0.7` would likely
appear. The SLM reached a reproducibility rate of **100%** across all three runs, as
expected a deterministic classification pipeline always produces the same embedding
and therefore the same label for a given input.

**A key observation: stable systematic errors.**
Sentences 4 and 5 (true label: `love`) and sentence 9 (true label: `fear`) were
predicted as `sadness` by the LLM across all 6 runs both temperatures, all three
repetitions. Sentence 4 *"i just want that feeling of not caring about unnecessary
stuff like i felt before"* and sentence 9 *"i have moments where i just feel so
overwhelmed that my eyes well up with tears"* both carry surface-level negative affect
that the LLM consistently interprets as `sadness`, regardless of the true label. This
is not a reproducibility failure: it is a **reproducible misclassification**. The model
is perfectly consistent in being wrong, confirming the `love`/`sadness` and
`fear`/`sadness` confusions already observed in P.1. For any production system, a
stable wrong prediction is just as harmful as a random one and harder to detect,
since it will not surface as variance in repeated calls.

---
## Section 5 — P.4 Prompt Sensitivity

We test three prompts on the same 30-sentence stratified sample (5 per class) at `temperature=0` to measure how prompt design affects LLM accuracy.

In [ ]:
# ── Stratified 30-sentence sample (5 per class) ───────────────────────────────
sample_30 = stratified_sample(test_ds, n_per_class=5, seed=7)
p4_texts, p4_true = zip(*sample_30)
print(f"Sample size: {len(p4_texts)} sentences")

Sample size: 30 sentences


### Prompt A — Minimal (same as P.1)

```
Classify the emotion expressed in the following sentence.
Choose exactly one from: sadness, joy, love, anger, fear, surprise.
Reply with a single word — the emotion label only.

Sentence: {sentence}
```

In [ ]:
# Prompt A — already defined as PROMPT_A
p4_a_preds = []
for text in tqdm(p4_texts, desc="Prompt A"):
    raw    = call_mistral(PROMPT_A.format(sentence=text), temperature=0.0)
    parsed = parse_emotion(raw)
    p4_a_preds.append(parsed)
    time.sleep(1)

Prompt A:   0%|          | 0/30 [00:00<?, ?it/s]

### Prompt B — Structured (definitions + examples)

```
Classify the emotion in the sentence below. Use exactly one of these labels:

- sadness: a feeling of sorrow or unhappiness. Example: "I miss my old friend so much."
- joy: a feeling of happiness or delight. Example: "I just got the best news of my life!"
- love: deep affection toward someone or something. Example: "I adore spending time with my family."
- anger: a feeling of displeasure or hostility. Example: "I am furious at how I was treated."
- fear: anxiety caused by perceived danger. Example: "The dark alley made me terrified."
- surprise: astonishment at something unexpected. Example: "I had no idea it was a surprise party!"

Reply with a single word — the emotion label only.

Sentence: {sentence}
```

In [ ]:
PROMPT_B = """Classify the emotion in the sentence below. Use exactly one of these labels:

- sadness: a feeling of sorrow or unhappiness. Example: "I miss my old friend so much."
- joy: a feeling of happiness or delight. Example: "I just got the best news of my life!"
- love: deep affection toward someone or something. Example: "I adore spending time with my family."
- anger: a feeling of displeasure or hostility. Example: "I am furious at how I was treated."
- fear: anxiety caused by perceived danger. Example: "The dark alley made me terrified."
- surprise: astonishment at something unexpected. Example: "I had no idea it was a surprise party!"

Reply with a single word — the emotion label only.

Sentence: {sentence}
"""

p4_b_preds = []
for text in tqdm(p4_texts, desc="Prompt B"):
    raw    = call_mistral(PROMPT_B.format(sentence=text), temperature=0.0)
    parsed = parse_emotion(raw)
    p4_b_preds.append(parsed)
    time.sleep(1)

Prompt B:   0%|          | 0/30 [00:00<?, ?it/s]

### Prompt C — Chain-of-thought

```
You will classify the emotion in a sentence step by step.

The possible emotions are: sadness, joy, love, anger, fear, surprise.

Think carefully about the tone, word choice, and context of the sentence.
Consider what the speaker is likely feeling and why.
Then write your final answer on a new line starting with: Answer:

Sentence: {sentence}
```

In [ ]:
PROMPT_C = """You will classify the emotion in a sentence step by step.

The possible emotions are: sadness, joy, love, anger, fear, surprise.

Think carefully about the tone, word choice, and context of the sentence.
Consider what the speaker is likely feeling and why.
Then write your final answer on a new line starting with: Answer:

Sentence: {sentence}
"""

p4_c_preds = []
for text in tqdm(p4_texts, desc="Prompt C"):
    raw    = call_mistral(PROMPT_C.format(sentence=text),
                          temperature=0.0, max_new_tokens=150)
    parsed = parse_emotion_after_answer(raw)
    p4_c_preds.append(parsed)
    time.sleep(1)

Prompt C:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
# ── P.4 — Comparison DataFrame ────────────────────────────────────────────────
def prompt_metrics(preds, true_labels):
    parseable = [(p, t) for p, t in zip(preds, true_labels) if p is not None]
    if not parseable:
        return 0.0, 0.0
    p_ids = [LABEL_IDS[p] for p, _ in parseable]
    t_ids = [t            for _, t in parseable]
    acc  = accuracy_score(t_ids, p_ids)
    pars = len(parseable) / len(preds)
    return acc, pars

acc_a, pars_a = prompt_metrics(p4_a_preds, p4_true)
acc_b, pars_b = prompt_metrics(p4_b_preds, p4_true)
acc_c, pars_c = prompt_metrics(p4_c_preds, p4_true)

p4_df = pd.DataFrame({
    "Prompt"        : ["A — Minimal", "B — Structured", "C — Chain-of-thought"],
    "Accuracy"      : [round(acc_a,3),  round(acc_b,3),  round(acc_c,3)],
    "Parseable %"   : [round(pars_a*100,1), round(pars_b*100,1), round(pars_c*100,1)],
})
print(p4_df.to_string(index=False))

              Prompt  Accuracy  Parseable %
         A — Minimal     0.600        100.0
      B — Structured     0.600        100.0
C — Chain-of-thought     0.364         36.7


### P.4 — Analysis

**Which prompt performed best?**
Prompts A and B achieved identical accuracy (**0.600**) with a **100% parseable rate**.
Prompt C (chain-of-thought) reached only **0.364** accuracy with a parseable rate of
**36.7%** meaning only 11 out of 30 responses could be mapped to a valid emotion label.
Prompts A and B therefore outperform Prompt C both in accuracy and in output reliability,
which is the opposite of the improvement chain-of-thought typically promises.

**What does this tell us about how the model works?**
The absence of any improvement from Prompt A to Prompt B is notable: adding explicit
per-class definitions and one example per category did not help the model discriminate
between emotion classes any better than the minimal prompt. This suggests that for a
large instruction-tuned model like `llama-3.3-70b-versatile`, the six emotion labels are
already sufficiently well-understood from pre-training, and additional definitional
context provides no extra signal at `temperature=0`.

Prompt C's results are the most informative. The drop in accuracy to **0.364** is partly
an artefact of the low parseable rate: with only **36.7%** of responses successfully
parsed, the effective evaluation sample shrinks to 11 sentences, making the estimate
unreliable. The root cause is that the chain-of-thought format encouraged long, free-form
reasoning traces that did not consistently follow the required `Answer: <label>` structure even though the instruction was explicit. This illustrates a key practical tension in
chain-of-thought prompting: **output format compliance degrades as response length
increases**, and parsing becomes the bottleneck rather than reasoning quality itself.

**Were there sentences where chain-of-thought made a visible difference?**
On the 11 successfully parsed responses, the reasoning traces occasionally identified
useful intermediate cues such as hedged phrasing or negation before settling on a
label. However, these isolated gains are more than offset by the 19 discarded responses.
In a real deployment, Prompt C would require a more robust parsing strategy a stricter
regex, a follow-up extraction call, or enforced structured output before its potential
reasoning benefits could be reliably captured.

## Section 6 — P.5 Deployment Recommendation

Based on the measurements in P.1 - P.4, we provide a justified recommendation for each
of three deployment scenarios.

---

### Scenario A — Real-time at scale
*Social media platform. 50 requests/second. Latency under 20 ms. Cost per request near zero.*

**Recommendation: SLM (DistilBERT)**

DistilBERT is the only viable choice here. Deployed locally, it runs as a deterministic
pipeline with sub-millisecond inference per sentence and zero per-call API cost both
requirements that the LLM cannot meet. The LLM depends on a remote API with non-trivial
round-trip latency, hard rate limits on the free tier, and a per-call cost that scales
linearly with volume, making it structurally incompatible with a 50 req/s throughput
target. The trade-off accepted is that ambiguous and intensity-driven cases which
represented 5 out of 10 sampled SLM errors in P.2 will occasionally be misclassified,
but the latency and cost constraints leave no alternative.

---

### Scenario B — High-value accuracy
*Mental health platform analysing patient notes. Accuracy is the priority. Results must
be reproducible.*

**Recommendation: SLM (DistilBERT) with mandatory human review on low-confidence outputs**

Despite the LLM's potential edge on nuanced language, the SLM is the stronger choice
here on both criteria. On accuracy, the SLM achieved a macro F1 of **0.8825** on 2 000
sentences (P.1), compared to **0.4350** for the LLM on its 300-sentence sample a gap
too large to accept in a high-stakes clinical setting. On reproducibility, the SLM
reached **100%** across all three runs (P.3), whereas the LLM's near-determinism at
`temperature=0` depends on remote infrastructure it does not control. More critically,
P.3 showed that the LLM produces **stable systematic errors**: sentences with true label
`love` or `fear` were consistently misclassified as `sadness` across all runs and both
temperatures a pattern that would silently skew any downstream clinical assessment.
The trade-off accepted is that the SLM will still misclassify some intensity and
negation cases (P.2); these should be flagged for human review rather than acted on
automatically.

---

### Scenario C — Conversational assistant
*Customer support chatbot. Must handle sarcasm, negation, and informal language. Natural
language output is useful for logging.*

**Recommendation: LLM with Prompt B (structured)**

This is the one scenario where the LLM's profile is a genuine fit. In P.2, the LLM's
errors were concentrated in Ambiguity (3/10) and Intensity (3/10), with only one
irony/sarcasm failure suggesting it handles pragmatically complex language more
gracefully than the SLM, whose errors skewed toward Intensity (4/10) and Other (3/10).
The LLM also natively produces natural language output that can be logged alongside
ticket text without post-processing. Prompt B which matched Prompt A's accuracy of
**0.600** on the 30-sentence sample while providing clearer label definitions — is
preferred over Prompt A for borderline cases, and over Prompt C whose parseable rate
of **36.7%** makes it operationally unreliable (P.4). The trade-offs accepted are
higher latency, API dependency at scale, and a substantially lower overall accuracy
than the SLM (macro F1 0.435 vs 0.883 in P.1).

## Section 7 — Conclusion

The central lesson of this investigation is that model choice is inseparable from
deployment context. DistilBERT's fine-tuned specialisation makes it faster, cheaper,
more accurate, and perfectly reproducible winning decisively in latency-critical and
audit-sensitive scenarios. Llama-3.3-70b-versatile's broader linguistic knowledge
makes it more robust to pragmatically complex language, but its zero-shot accuracy
(macro F1 0.435 vs 0.883) and dependency on remote infrastructure limit its usefulness
to scenarios where nuance and natural language output matter more than raw performance.
The most unexpected finding was not the accuracy gap itself, but the nature of the LLM's
errors: at both `temperature=0` and `temperature=0.7`, misclassifications were entirely
stable across runs, showing that **reproducibility and correctness are independent
properties** a model can be perfectly consistent and systematically wrong at the same
time. Any evaluation framework that measures only variance, without measuring accuracy,
will miss this failure mode entirely.